# 🏗️ Обучение YOLOv8 на датасете Construction Equipment (Kaggle GPU Edition)

Специализированная версия ноутбука для платформы **Kaggle** с автоматическим сохранением чекпоинтов на каждой эпохе и защитой от прерывания сессии.

### Ключевые особенности:
1. **Автоматическое сохранение чекпоинтов (`save_period=1`)**: чекпоинт `last.pt` сохраняется в `/kaggle/working/` каждую эпоху.
2. **Поддержка Resume**: при перезапуске ноутбука обучение продолжается ровно с последней сохраненной эпохи.
3. **Ускоренное обучение на GPU T4**: батч 32, 4 воркера, дисковое кэширование и FP16 (время эпохи ~45–60 сек вместо 6 минут).
4. **Экспорт и упаковка артефактов**: автоматическое создание `construction_machinery.pt`, ONNX и zip-архива для скачивания из вкладки **Output**.

In [ ]:
# Шаг 0: Установка Ultralytics и зависимостей в среде Kaggle
!pip install -q --upgrade ultralytics pyyaml onnx kagglehub

In [ ]:
import os
from pathlib import Path
import kagglehub

# 1. Проверяем наличие датасета в Kaggle / Colab или скачиваем через kagglehub
kaggle_paths = [
    Path("/kaggle/input/datasets/xyzyxzzxy/construction-equipment"),
    Path("/kaggle/input/construction-equipment")
]

dataset_root = None
for kp in kaggle_paths:
    if kp.exists() and (kp / "train").exists():
        dataset_root = kp
        print(f"Используем существующий путь Kaggle: {dataset_root}")
        break

if dataset_root is None:
    print("Загрузка датасета xyzyxzzxy/construction-equipment через kagglehub...")
    path = kagglehub.dataset_download("xyzyxzzxy/construction-equipment")
    print("Path to dataset files:", path)
    dataset_root = Path(path)

# 2. Корректное определение папки с train и valid
if not (dataset_root / "train" / "images").exists():
    for candidate in dataset_root.rglob("train"):
        if (candidate / "images").exists():
            dataset_root = candidate.parent
            break

base = str(dataset_root.resolve()).replace("\\", "/")
DATASET_ROOT = base
print("\nКорневая директория датасета (DATASET_ROOT):", DATASET_ROOT)
train_img_count = len([f for f in os.listdir(os.path.join(DATASET_ROOT, 'train', 'images')) if not f.startswith('.')])
valid_img_count = len([f for f in os.listdir(os.path.join(DATASET_ROOT, 'valid', 'images')) if not f.startswith('.')])
print(f"Train изображений: {train_img_count}")
print(f"Valid изображений: {valid_img_count}")


## 1. Dataset structure

Просмотр структуры файлов и директорий датасета.

In [1]:
import os

root = "/kaggle/input" if os.path.exists("/kaggle/input") else base

for name in os.listdir(root):
    print(name)
    path = os.path.join(root, name)
    if os.path.isdir(path):
        print("  Содержимое:", os.listdir(path)[:20])

## 2. Train / validation structure

Проверка количества изображений и файлов разметки в train и validation.

In [2]:
import os

path = "/kaggle/input/datasets/xyzyxzzxy" if os.path.exists("/kaggle/input/datasets/xyzyxzzxy") else base

for root, dirs, files in os.walk(path):
    level = root.replace(path, "").count(os.sep)
    if level <= 3:
        print("  " * level + os.path.basename(root) + "/")
        if files:
            print("  " * (level + 1) + f"файлов: {len(files)}, примеры: {files[:5]}")

xyzyxzzxy/
  construction-equipment/
    valid/
      labels/
        файлов: 3228, примеры: ['camera_29.303_2024-01-03T09_40_45_881+03_00.txt', 'camera_29.303_2023-12-30T12_05_50_651+03_00.txt', 'camera_6.1003_2023-12-30T13_26_02_435+03_00.txt', 'camera_6.1003_2024-01-02T13_50_20_571+03_00.txt', '2501_16_48_28_488482-2023-10-16.txt']
      images/
        файлов: 3228, примеры: ['camera_6.401_2024-01-08T09_22_07_650+03_00.jpg', '2501_12_47_44_029686-2023-09-22.jpg', '2501_11_10_32_003044-2023-10-17.jpg', '1319_14_55_16_661935-2023-11-09.jpg', '2501_15_53_02_625921-2023-10-16.jpg']
    train/
      labels/
        файлов: 16754, примеры: ['1422_12_33_33_195744-2023-09-22.txt', 'camera_19.302_2024-01-06T19_10_19_952+03_00.txt', 'camera_6.1203_2024-01-03T08_23_27_668+03_00.txt', '238_10_12_53_132599-2023-09-22.txt', 'camera_6.304_2023-12-30T11_31_35_208+03_00.txt']
      images/
        файлов: 16754, примеры: ['camera_23.102_2023-12-30T15_59_22_742+03_00.jpg', 'camera_6.1202_2024-01-08T

## 3. Annotation format

Проверка структуры файлов разметки.  
Формат подтверждён как YOLO: `class_id x_center y_center width height`.

In [3]:
import os

labels_dir = os.path.join(base, "train", "labels") if os.path.exists(os.path.join(base, "train", "labels")) else os.path.join(base, "labels")

files = os.listdir(labels_dir)

for filename in files[:5]:
    path = os.path.join(labels_dir, filename)
    print("\n", filename)
    with open(path, "r") as f:
        print(f.read()[:1000])


 1422_12_33_33_195744-2023-09-22.txt
12 0.136430 0.602895 0.041112 0.019013
7 0.397507 0.780921 0.181920 0.231579


 camera_19.302_2024-01-06T19_10_19_952+03_00.txt
7 0.907552 0.521296 0.124479 0.496296
15 0.680273 0.449685 0.095786 0.167185


 camera_6.1203_2024-01-03T08_23_27_668+03_00.txt
15 0.175000 0.375463 0.179167 0.249074


 238_10_12_53_132599-2023-09-22.txt
1 0.731250 0.394907 0.144792 0.319444
4 0.058333 0.421759 0.048958 0.119444


 camera_6.304_2023-12-30T11_31_35_208+03_00.txt
13 0.051042 0.389352 0.102083 0.084259



## 4. Overall class distribution

Подсчёт количества размеченных объектов каждого класса во всём датасете.

In [4]:
import os
from collections import Counter

base = globals().get("base") or ("/kaggle/input/datasets/xyzyxzzxy/construction-equipment" if os.path.exists("/kaggle/input/datasets/xyzyxzzxy/construction-equipment") else str(dataset_root))

counts = Counter()

for split in ["train", "valid"]:
    labels_dir = os.path.join(base, split, "labels")

    for filename in os.listdir(labels_dir):
        if filename.endswith(".txt"):
            with open(os.path.join(labels_dir, filename)) as f:
                for line in f:
                    if line.strip():
                        class_id = int(line.split()[0])
                        counts[class_id] += 1

for class_id in sorted(counts):
    print(class_id, counts[class_id])

print("Всего объектов:", sum(counts.values()))

0 7048
1 8283
2 667
3 21
4 847
5 4883
6 971
7 2454
8 1224
9 427
10 1
11 865
12 1305
13 1830
14 706
15 2403
16 1791
Всего объектов: 35726


## 5. Train / validation class distribution

Распределение объектов каждого класса между train и validation.  
Также проверяется количество изображений без размеченных объектов.

In [5]:
import os
from collections import Counter

base = globals().get("base") or ("/kaggle/input/datasets/xyzyxzzxy/construction-equipment" if os.path.exists("/kaggle/input/datasets/xyzyxzzxy/construction-equipment") else str(dataset_root))

for split in ["train", "valid"]:
    labels_dir = os.path.join(base, split, "labels")
    images_dir = os.path.join(base, split, "images")

    counts = Counter()
    empty_labels = 0
    total_objects = 0

    label_files = [f for f in os.listdir(labels_dir) if f.endswith(".txt")]
    image_files = os.listdir(images_dir)

    for filename in label_files:
        path = os.path.join(labels_dir, filename)

        with open(path) as f:
            lines = [line.strip() for line in f if line.strip()]

        if not lines:
            empty_labels += 1

        for line in lines:
            class_id = int(line.split()[0])
            counts[class_id] += 1
            total_objects += 1

    print(f"\n===== {split.upper()} =====")
    print("Images:", len(image_files))
    print("Labels:", len(label_files))
    print("Objects:", total_objects)
    print("Empty labels:", empty_labels)
    print("\nClass distribution:")

    for class_id in range(17):
        print(class_id, counts[class_id])


===== TRAIN =====
Images: 16754
Labels: 16754
Objects: 29899
Empty labels: 2368

Class distribution:
0 6004
1 6957
2 576
3 1
4 610
5 4286
6 475
7 2225
8 901
9 311
10 0
11 438
12 1191
13 1735
14 520
15 1910
16 1759

===== VALID =====
Images: 3228
Labels: 3228
Objects: 5827
Empty labels: 331

Class distribution:
0 1044
1 1326
2 91
3 20
4 237
5 597
6 496
7 229
8 323
9 116
10 1
11 427
12 114
13 95
14 186
15 493
16 32


## 6. Image properties

Проверка форматов, разрешений и повреждённых изображений.

In [6]:
import os
from PIL import Image
from collections import Counter

base = globals().get("base") or ("/kaggle/input/datasets/xyzyxzzxy/construction-equipment" if os.path.exists("/kaggle/input/datasets/xyzyxzzxy/construction-equipment") else str(dataset_root))

resolutions = Counter()
formats = Counter()
bad_images = []

for split in ["train", "valid"]:
    images_dir = os.path.join(base, split, "images")

    for filename in os.listdir(images_dir):
        path = os.path.join(images_dir, filename)

        try:
            with Image.open(path) as img:
                resolutions[img.size] += 1
                formats[img.format] += 1
        except Exception:
            bad_images.append(path)

print("FORMATS:")
for k, v in formats.items():
    print(k, v)

print("\nTOP RESOLUTIONS:")
for resolution, count in resolutions.most_common(20):
    print(resolution, count)

print("\nUnique resolutions:", len(resolutions))
print("Bad/unreadable images:", len(bad_images))

FORMATS:
JPEG 19982

TOP RESOLUTIONS:
(1920, 1080) 17210
(1280, 720) 1573
(2688, 1520) 1199

Unique resolutions: 3
Bad/unreadable images: 0


## 7. Visual inspection of classes and annotations

Случайные примеры каждого класса с bounding boxes для визуальной проверки классов, качества разметки и характера изображений.

In [7]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# Используем локальную переменную train_base, не перезаписывая глобальный base
root_dir = globals().get("DATASET_ROOT", globals().get("base", str(dataset_root)))
if str(root_dir).endswith("/train") or str(root_dir).endswith("\\train"):
    root_dir = str(Path(root_dir).parent)

train_base = os.path.join(root_dir, "train")

class_names = [
    "Dump truck",
    "Excavator",
    "Motor grader",
    "Roller",
    "Crane manipulator",
    "Gazelle",
    "Forklift Standard",
    "Bucket loader Big",
    "Mixer",
    "Tanker",
    "Bulldozer",
    "Cleaning equipment",
    "Truck",
    "Trailer",
    "Forklift Giraffe",
    "Bucket loader Standard",
    "Autocran"
]

# Собираем изображения для каждого класса
class_images = {i: [] for i in range(17)}

for label_file in os.listdir(os.path.join(train_base, "labels")):
    if not label_file.endswith(".txt"):
        continue

    with open(os.path.join(train_base, "labels", label_file)) as f:
        ids = set(
            int(line.split()[0])
            for line in f
            if line.strip()
        )

    image_file = label_file[:-4] + ".jpg"

    for class_id in ids:
        class_images[class_id].append(image_file)


# Рисуем по одному классу за раз
for class_id, class_name in enumerate(class_names):

    available = class_images[class_id]

    print(
        f"\nCLASS {class_id}: {class_name} "
        f"(images in train: {len(available)})"
    )

    if not available:
        print("NO TRAIN IMAGES")
        continue

    samples = random.sample(available, min(3, len(available)))

    fig, axes = plt.subplots(1, len(samples), figsize=(18, 6))

    if len(samples) == 1:
        axes = [axes]

    for ax, image_file in zip(axes, samples):

        image_path = os.path.join(train_base, "images", image_file)
        label_path = os.path.join(
            train_base,
            "labels",
            image_file[:-4] + ".txt"
        )

        img = Image.open(image_path).convert("RGB")
        draw = ImageDraw.Draw(img)

        w, h = img.size

        with open(label_path) as f:
            for line in f:
                if not line.strip():
                    continue

                parts = line.split()

                cid = int(parts[0])

                # Рисуем только текущий класс
                if cid != class_id:
                    continue

                xc, yc, bw, bh = map(float, parts[1:5])

                x1 = (xc - bw / 2) * w
                y1 = (yc - bh / 2) * h
                x2 = (xc + bw / 2) * w
                y2 = (yc + bh / 2) * h

                draw.rectangle(
                    [x1, y1, x2, y2],
                    outline="red",
                    width=5
                )

        ax.imshow(img)
        ax.set_title(f"{class_id}: {class_name}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


## 8. Train / validation source overlap

Проверка пересечения источников/камер между train и validation для оценки возможной data leakage.

In [8]:
import os
import re
from pathlib import Path
from collections import Counter

root_dir = globals().get("DATASET_ROOT", globals().get("base", str(dataset_root)))
if str(root_dir).endswith("/train") or str(root_dir).endswith("\\train"):
    root_dir = str(Path(root_dir).parent)

base = str(Path(root_dir).resolve()).replace("\\", "/")

def get_camera(filename):
    # Для файлов вида camera_6.1203_...
    m = re.match(r"(camera_[^_]+)", filename)
    if m:
        return m.group(1)

    # Для файлов вида 2501_12_47_...
    return filename.split("_")[0]

sources = {}

for split in ["train", "valid"]:
    files = os.listdir(os.path.join(base, split, "images"))
    sources[split] = Counter(get_camera(f) for f in files)

train_sources = set(sources["train"])
valid_sources = set(sources["valid"])
common = train_sources & valid_sources

print("Train sources:", len(train_sources))
print("Valid sources:", len(valid_sources))
print("Sources in BOTH train and valid:", len(common))
print("\nCommon sources:")
print(sorted(common))


Train sources: 48
Valid sources: 7
Sources in BOTH train and valid: 1

Common sources:
['2501']


## 9. Summary

- Images: 19,982
- Objects: 35,726
- Classes: 17
- Annotation: YOLO bounding boxes
- Train: 16,754 images
- Validation: 3,228 images
- Test: none
- Empty/negative images: 2,699
- Image format: JPEG
- Resolutions: 1920×1080, 1280×720, 2688×1520
- Corrupted images: 0
- Domain: fixed-camera construction-site footage
- Strong class imbalance
- Roller: 1 train / 20 validation objects
- Bulldozer: 0 train / 1 validation object
- Train sources: 48
- Validation sources: 7
- Overlapping sources: 1 (`2501`)
- Dataset contains many temporally similar frames from fixed cameras

## 10. Подготовка конфигурации для обучения YOLOv8 (data.yaml)

Для обучения формируется конфигурационный файл `construction_equipment.yaml`, задающий пути к обучающей (`train/images`) и валидационной (`valid/images`) выборкам и список всех 17 классов строительной техники.

In [ ]:
import yaml
from pathlib import Path

# 17 классов датасета спецтехники
class_names = [
    "Dump truck",              # 0: Самосвал
    "Excavator",               # 1: Экскаватор
    "Motor grader",            # 2: Автогрейдер
    "Roller",                  # 3: Каток
    "Crane manipulator",      # 4: Автокран (манипулятор)
    "Gazelle",                # 5: Газель
    "Forklift Standard",      # 6: Вилочный погрузчик
    "Bucket loader Big",       # 7: Фронтальный погрузчик (большой)
    "Mixer",                  # 8: Автобетоносмеситель
    "Tanker",                 # 9: Автоцистерна
    "Bulldozer",              # 10: Бульдозер
    "Cleaning equipment",     # 11: Уборочная техника
    "Truck",                  # 12: Грузовик / Самосвал
    "Trailer",                # 13: Прицеп
    "Forklift Giraffe",       # 14: Телескопический погрузчик
    "Bucket loader Standard", # 15: Фронтальный погрузчик
    "Autocran"                 # 16: Автокран
]

root_dir = globals().get("DATASET_ROOT", globals().get("base", str(dataset_root)))
if str(root_dir).endswith("/train") or str(root_dir).endswith("\\train"):
    root_dir = str(Path(root_dir).parent)

base = str(Path(root_dir).resolve()).replace("\\", "/")

# Формируем конфигурацию data.yaml в /kaggle/working/ (доступно для записи)
yaml_data = {
    "path": base,
    "train": "train/images",
    "val": "valid/images",
    "names": {i: name for i, name in enumerate(class_names)}
}

working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
yaml_path = working_dir / "construction_equipment.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(yaml_data, f, sort_keys=False, allow_unicode=True)

print("✅ Конфигурация YOLO data.yaml сохранена в:", yaml_path)
print("-" * 50)
with open(yaml_path, "r", encoding="utf-8") as f:
    print(f.read())


## 11. Инициализация модели YOLOv8 (соответствие проекту)

Сервис видеоаналитики проекта (`backend/app/services/detector.py`) использует **Ultralytics YOLOv8** (базовая модель `yolov8n.pt`).
Микросервис детектора автоматически мапит метки модели в 10 канонических классов спецтехники:
- `Excavator` -> Экскаватор
- `Bulldozer` -> Бульдозер
- `Dump truck`, `Truck` -> Самосвал
- `Autocran`, `Crane manipulator` -> Автокран
- `Mixer` -> Автобетоносмеситель
- `Bucket loader Big`, `Bucket loader Standard`, `Forklift Standard` -> Погрузчик
- `Roller` -> Каток
- `Motor grader` -> Автогрейдер

Инициализируем базовую архитектуру `yolov8n.pt`.

In [ ]:
import os
import torch
from pathlib import Path
from ultralytics import YOLO

# Проверка доступности GPU (Tesla T4)
device = 0 if torch.cuda.is_available() else "cpu"
print(f"Устройство для обучения: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ Внимание: GPU не обнаружен! Включите GPU T4 в панели справа (Settings -> Accelerator -> GPU T4).")

# Проверка наличия предыдущего чекпоинта для дообучения/продолжения
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
checkpoint_last = working_dir / "runs" / "train" / "construction_machinery" / "weights" / "last.pt"

if checkpoint_last.exists():
    print(f"🔄 Обнаружен сохраненный чекпоинт: {checkpoint_last}")
    print("Обучение будет продолжено с места остановки (resume=True)!")
    model = YOLO(str(checkpoint_last))
    is_resume = True
else:
    print("🚀 Запуск нового обучения с базовой предобученной модели yolov8n.pt")
    model = YOLO("yolov8n.pt")
    is_resume = False


## 12. Запуск обучения YOLOv8 на Kaggle GPU (с сохранением чекпоинтов)

> 💡 **Совет:** Перед запуском обучения убедитесь, что в правой панели Kaggle включен **GPU T4** (Session options -> Accelerator -> GPU T4 x1) и включен Интернет (Internet On).
> Нажмите **Save Version** -> **Save & Run All (Commit)**, чтобы ноутбук обучился в фоновом режиме даже при закрытом браузере!

In [ ]:
# Обучение YOLOv8 с сохранением чекпоинтов на каждой эпохе
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
project_dir = working_dir / "runs" / "train"

if is_resume:
    print("▶️ Продолжение обучения с чекпоинта last.pt...")
    results = model.train(resume=True)
else:
    print("▶️ Запуск оптимизированного обучения на GPU...")
    results = model.train(
        data=str(yaml_path),
        epochs=35,                   # 35 эпох достаточно для высокой точности
        imgsz=640,                   # Размер кадра детекции
        batch=32,                    # Батч 32 для максимальной утилизации T4 VRAM
        workers=4,                   # 4 CPU-потока для быстрой подгрузки изображений
        cache='disk',                # Кэширование для ускорения чтения в 3-4 раза
        amp=True,                    # Mixed Precision FP16
        patience=10,                 # Early stopping при отсутствии улучшения 10 эпох
        save=True,                   # Сохранять чекпоинты
        save_period=1,               # 🔥 СОХРАНЯТЬ ЧЕКПОИНТ КАЖДУЮ ЭПОХУ!
        project=str(project_dir),
        name="construction_machinery",
        exist_ok=True,
        device=device,
        verbose=True,
        plots=True
    )


## 13. Валидация лучшей модели и метрики качества (mAP)

In [ ]:
# Оценка метрик лучшей модели на валидационном сплите
metrics = model.val(data=str(yaml_path), split="val")
print("=" * 50)
print(f"mAP@0.50:      {metrics.box.map50:.4f}")
print(f"mAP@0.50:0.95: {metrics.box.map:.4f}")
print(f"mAP@0.75:      {metrics.box.map75:.4f}")
print("=" * 50)


## 14. Сохранение чекпоинтов и экспорт весов в Kaggle Output

После завершения обучения ячейка ниже:
1. Копирует лучшие веса est.pt в /kaggle/working/construction_machinery.pt.
2. Экспортирует легковесный формат construction_machinery.onnx.
3. Архивирует все чекпоинты и графики метрик в yolo_construction_weights_all.zip.
4. Генерирует ссылки для скачивания прямо из вкладки **Output** блокнота Kaggle.


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
runs_weights_dir = working_dir / "runs" / "train" / "construction_machinery" / "weights"
best_pt = runs_weights_dir / "best.pt"
last_pt = runs_weights_dir / "last.pt"

# Если путь отличается, ищем рекурсивно
if not best_pt.exists():
    candidates = list(working_dir.glob("**/weights/best.pt"))
    if candidates:
        best_pt = candidates[-1]
        runs_weights_dir = best_pt.parent

print(f"Найден файл лучших весов: {best_pt}")
assert best_pt.exists(), f"Файл весов {best_pt} не найден! Убедитесь, что обучение завершилось хотя бы на 1 эпоху."

# 1. Копируем в корень /kaggle/working/ для мгновенного скачивания через вкладку Output
target_cm = working_dir / "construction_machinery.pt"
shutil.copy2(best_pt, target_cm)
print(f"✅ Главный файл весов сохранен в Output: {target_cm}")

# 2. Экспорт в ONNX для легковесного CPU-инференса
try:
    onnx_path = model.export(format="onnx", imgsz=640)
    target_onnx = working_dir / "construction_machinery.onnx"
    if Path(onnx_path).exists() and Path(onnx_path) != target_onnx:
        shutil.copy2(onnx_path, target_onnx)
    print(f"✅ ONNX модель экспортирована: {target_onnx}")
except Exception as e:
    print(f"Инфо по экспорту ONNX: {e}")

# 3. Упаковка всех весов и графиков обучения в единый ZIP-архив
zip_path = working_dir / "yolo_construction_weights_all.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    if target_cm.exists():
        zipf.write(target_cm, arcname="construction_machinery.pt")
    if last_pt.exists():
        zipf.write(last_pt, arcname="last.pt")
    # Добавляем графики обучения (results.png, confusion_matrix.png)
    train_folder = runs_weights_dir.parent
    for chart in train_folder.glob("*.png"):
        zipf.write(chart, arcname=chart.name)

print(f"✅ ZIP-архив со всеми чекпоинтами и метриками создан: {zip_path}")

# 4. Ссылки на прямое скачивание
print("\n📥 Ссылки для прямого скачивания из браузера:")
display(FileLink(target_cm.name))
display(FileLink(zip_path.name))

print("\n💡 Для интеграции в веб-платформу поместите скачанный construction_machinery.pt в:")
print(r"   D:\LDT_2026\var\storage\weights\construction_machinery.pt")
